<a href="https://colab.research.google.com/github/sebmca/Student-Drug-Addiction-Prediction/blob/main/Drug_Addiction_GITHUB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## The following !cp is used to test the following COLAB code in GITHUB

# !cp "/content/drive/MyDrive/Drug_Addiction_Data.csv" "/content/Drug_Addiction_Data.csv"
# !cp -r "/content/drive/MyDrive/cols_indexed_drug_addiction" "/content/cols_indexed_drug_addiction"
# !cp "/content/drive/MyDrive/model_drug_addiction.cbm" "/content/model_drug_addiction.cbm"
# !cp "/content/drive/MyDrive/Bulk_Drug_Data.csv" "/content/Bulk_Drug_Data.csv"
# !cp "/content/drive/MyDrive/Single_Input_Drug.csv" "/content/Single_Input_Drug.csv"


## OVAL shape like Main Dashboard

#### COPY THIS IN CODE CELL 1

# The dataset is collected from different Sources, they are

# 1. UCI [University of California, Irvine], Central Excise and
#    from different De-Addiction centres

# 2. Total 75,000 records and 19 columns are in the Dataset

# 3. The columns contain,

#    StudentID,ALT_Blood,GGT_Blood,Glucose_Blood,Creatine_Blood,
#    THC_Positive_Urine,Cocaine_Positive_Urine,Opioid_Positive_Urine,Urine_pH,
#    Attendance,Marks_drop,Assignment_delays,Frequent_leave,Mood_Swings,
#    Peer_Group_Issues,Disciplinary_Actions,Counseling_Referral,Social_Media_Overuse,
#    Gaming_Addiction,Addiction_Label

# Import Javascript and HTML for front end GUI design

from IPython.display import display, HTML,Javascript
from ipywidgets import VBox, HBox
import ipywidgets as widgets
from IPython.display import clear_output

# Define a function to scroll your code cell directly to output grid, other wise you will
# have to manually

def set_focus_cell1():
    display(Javascript("""
        document.getElementById('output-area').scrollIntoView({behavior: 'smooth'});
    """))

# Call set_focus_cell1 to scroll down automatically this cell

set_focus_cell1()

display(HTML("<h3 style='color:green; text-align:left;'> \
  Loading Libraries Please Wait !!! </h3>"))
print()

# To display menu images image base64 is needed
import base64
import PIL

# To get the Processing time of a particular operation import
# the following

from datetime import datetime

# !mv "/content/drive/MyDrive/Colab Notebooks/Drug_Addiction_GITHUB.ipynb" "/content/drive/MyDrive" 2>/dev/null

## Import CATBOOST & PYSPARK related libraries

!pip install -q  catboost pyspark

import pandas as pd
import sklearn

import numpy as np
import argparse

import catboost
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, \
                              recall_score, f1_score

from sklearn.metrics import confusion_matrix
import seaborn as sns

# To display images and graphs
import matplotlib.pyplot as plt

# Dataset Manipulation
import os

# File Upload package import
from google.colab import files

# Used to suppress user warning while display graphs and plots

import warnings

## Import Machine Learning Libraries and Packages

import pyspark

from pyspark.ml.classification import LogisticRegression
# Accuracy Evluators
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
## For Correlation Analysis
from pyspark.ml.feature import VectorAssembler, StringIndexer, IndexToString
from pyspark.ml import Pipeline

from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnull, regexp_replace, trim, when
import matplotlib.pyplot as plt # used for graph plotting

from pyspark.ml.classification import RandomForestClassifier, RandomForestClassificationModel

from pyspark.ml.classification import GBTClassifier
from pyspark.ml.classification import GBTClassificationModel

# To plot graphs import the followng packages

# Seaborn, a statistical data visualization library in Python, is commonly used
# in machine learning for various purposes.

# Co_relation package import

from pyspark.ml.stat import Correlation

# Import GBT Regression Packages

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import GBTRegressor, GBTRegressionModel
from pyspark.sql.functions import col, when, format_number, regexp_replace, expr, round
## to get null value count of each column
from pyspark.sql.functions import sum as _sum

# Create an instance of pyspark as follows:-
my_spark = SparkSession.builder.getOrCreate()

display(HTML("<style>.container { width: 100% !important; height: 500% !important; }</style>"))
print()
print()
print("PySpark version:", pyspark.__version__)
print("pandas version:", pd.__version__)
print("scikit-learn version:", sklearn.__version__)
# print("Seaborn version:", sns.__version__)
print("CatBoost version:", catboost.__version__)

## CLONE GITHUB REPOSITORY - IMPORTANT

!git clone https://github.com/sebmca/Student-Drug-Addiction-Prediction.git

display(HTML("<h3 style='color:orange; text-align:left;'> \
  All Packages Downloaded Successfully !!! </h3>"))
print()



In [ ]:

## COPY THE FOLLOWING IN CODE CELL 2

def set_focus_cell2():
    display(Javascript("""
        document.getElementById('output-area').scrollIntoView({behavior: 'smooth'});
    """))

# Call set_focus to scroll down automatically this cell

set_focus_cell2()

# Clear output area Fn

def clear_result_grid():
    clear_output(wait=True)

def load_data(event):
  print()
  clear_result_grid()
  set_focus_cell2()

  display(HTML("<h3 style='color:orange; text-align:left;'> \
    Reading Historical Dataset, this may take few seconds...wait !!!</h3>"))

# Record the start time (current_time) before the process starts

  current_time = datetime.now()
  print("Current Time :", current_time.strftime("%I:%M:%S %p"))  # Format the datetime

  global df_from_csv

# Read and Load your data

  df_from_csv = my_spark.read \
                  .format("csv") \
                  .option("header", "True") \
                  .option("inferSchema", "True") \
                  .load("/content/Student-Drug-Addiction-Prediction/Drug_Addiction_Data.csv")

# Cross Verification
  print("Schema & Summary of df_from_CSV")
  df_from_csv.printSchema()

  print("Dispaly few data in df_from_CSV")
  df_from_csv.show(5, False)

  tot_count = df_from_csv.count()
  print("Total number of Records = ",tot_count)

# Record the end time after the process has finished

  end_time = datetime.now()
  print("End Time H:M:S :", end_time.strftime("%I:%M:%S %p"))  # Format the datetime

# Calculate the time difference
  time_difference = end_time - current_time

# Print the time difference
  formatted_time_difference = str(time_difference).split('.')[0]  # Remove microseconds
  print("Loading Time :", formatted_time_difference)

  print()
  display(HTML("<h3 style='color:magenta; text-align:left;'> \
    All your Data loaded Successfully...</h3>"))

  go_back_function(event)

def EDA_start(event):

  print()
  clear_result_grid()
  set_focus_cell2()

  display(HTML("<h3 style='color:orange; text-align:left;'>EDA Started, Please wait !!!</h3>"))
  print()

  global score_cols

# Display Statistics of dataframe - EDA_1

  print("Statistics of df_from_CSV")
  df_from_csv.describe().show()
# Statistics of df_from_CSV
# +-------+---------+------------------+-----------------+------------------+------------------+------------------+----------------------+---------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+-------------------+--------------------+-------------------+--------------------+------------------+---------------+
# |summary|StudentID|         ALT_Blood|        GGT_Blood|     Glucose_Blood|    Creatine_Blood|THC_Positive_Urine|Cocaine_Positive_Urine|Opioid_Positive_Urine|          Urine_pH|         Attendance|        Marks_drop|  Assignment_delays|     Frequent_leave|       Mood_Swings|  Peer_Group_Issues|Disciplinary_Actions|Counseling_Referral|Social_Media_Overuse|  Gaming_Addiction|Addiction_Label|
# +-------+---------+------------------+-----------------+------------------+------------------+------------------+----------------------+---------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+-------------------+--------------------+-------------------+--------------------+------------------+---------------+
# |  count|    75000|             75000|            75000|             75000|             75000|             75000|                 75000|                75000|             75000|              75000|             75000|              75000|              75000|             75000|              75000|               75000|              75000|               75000|             75000|          75000|
# |   mean|     NULL|           59.3624|81.57602666666666|         110.38892|1.0999677333333293|              NULL|                  NULL|                 NULL| 5.996526666666592|            0.20176|0.2541466666666667|0.29458666666666666|0.15426666666666666|           0.50004|0.49809333333333333|  0.5044533333333333| 0.5017333333333334|  0.5017466666666667|           0.49884|           NULL|
# | stddev|     NULL|24.795013681707143|46.29872943804459|27.480943650654567|0.2926452497058928|              NULL|                  NULL|                 NULL|0.2991622680024838|0.40131664530725064|0.4353833551455185| 0.4558597736848442|0.36120659161332347|0.5000033317666563| 0.4999996979514382| 0.49998350066334224| 0.5000003288931643|  0.5000002824925753|0.5000019877558863|           NULL|
# |    min|   S00101|                10|               10|                70|               0.5|                No|                    No|                   No|               4.9|                  0|                 0|                  0|                  0|                 0|                  0|                   0|                  0|                   0|                 0|       Addicted|
# |    max|   S05100|               134|              258|               200|               2.0|               Yes|                   Yes|                  Yes|               7.1|                  1|                 1|                  1|                  1|                 1|                  1|                   1|                  1|                   1|                 1|   Not Addicted|
# +-------+---------+------------------+-----------------+------------------+------------------+------------------+----------------------+---------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+-------------------+--------------------+-------------------+--------------------+------------------+---------------+

## Graph_1 & Graph_2

# Step 1: Group and count THC_Positive_Urine values

  print()
  print()

  thc_counts = df_from_csv.groupBy("THC_Positive_Urine").count().orderBy("THC_Positive_Urine")

  print("Grouped counts of THC_Positive_Urine...")
  thc_counts.show()

# Step 2: Convert to Pandas DataFrame
  thc_pd = thc_counts.toPandas()

# Strip Plot (Dot distribution) - Equivalent to scatter plot
# hue - Each point will be colored based on whether it is "Yes" or "No"
  plt.figure(figsize=(6, 4))
  sns.stripplot(x='THC_Positive_Urine', y='count', hue='THC_Positive_Urine',
                data=thc_pd, palette='Set2', legend=False, size=10)  # 👈 increase dot size here
  plt.title('Dot Distribution of THC_Positive_Urine Counts')
  plt.xlabel('THC_Positive_Urine')
  plt.ylabel('Count')
# Add dark red grid lines
  plt.grid(True, axis='both', color='lightblue', linestyle='--', linewidth=1.2)
  plt.tight_layout()
  plt.show()

  print()
  print()
  print()

# Step 1: Group and count Cocaine_Positive_Urine values - POINT PLOT GRAPH

  cocaine_counts = df_from_csv.groupBy("Cocaine_Positive_Urine").count().orderBy("Cocaine_Positive_Urine")

  print("Grouped counts of Cocaine_Positive_Urine")
  cocaine_counts.show()

# Step 2: Convert to Pandas DataFrame

  cocaine_pd = cocaine_counts.toPandas()

  plt.figure(figsize=(6, 4))
  sns.pointplot(x='Cocaine_Positive_Urine', y='count', data=cocaine_pd)
  plt.title('Trend of Cocaine_Positive_Urine Counts')
  plt.xlabel('Cocaine_Positive_Urine')
  plt.ylabel('Count')
# Add dark red grid lines
  plt.grid(True, axis='both', color='lightblue', linestyle='--', linewidth=1.2)
  plt.tight_layout()
  plt.show()

  print()
  print()
  print()

# MOSAIC PLOT

  from statsmodels.graphics.mosaicplot import mosaic
  mosaic(cocaine_pd.set_index('Cocaine_Positive_Urine')['count'].to_dict())
  plt.title('Mosaic Plot of Cocaine_Positive_Urine')
  plt.show()
  print()
  print()
  print()

## DUMB-BELL PLOT showing all THC_Positive_Urine,Cocaine_Positive_Urine,
## Opioid_Positive_Urine

# Convert only the required columns to Pandas
  df_pd = df_from_csv.select("THC_Positive_Urine", "Cocaine_Positive_Urine",
                             "Opioid_Positive_Urine").toPandas()

# Melt the wide DataFrame into long format
  df_melted = df_pd.melt(var_name='Urine_Result', value_name='Result')  # Result will be 'Yes' or 'No'

# Group by drug type and result to get count
  dumbbell_df = df_melted.groupby(['Urine_Result', 'Result']).size().reset_index(name='Count')

# Pivot for dumbbell plot (Yes and No as columns)
  dumbbell_pivot = dumbbell_df.pivot(index='Urine_Result', columns='Result',
                                     values='Count').reset_index()

## CROSS VERIFICATION -PRINT SUMMARY of each urine test

  print("Summary of each counts in Urine Test ....")
  print(dumbbell_pivot)

  # for index, row in dumbbell_pivot.iterrows():
  #     print(f"{row['Urine_Result']}: No = {row['No']}, Yes = {row['Yes']}")

## Start Plotting the Graph

# import matplotlib.pyplot as plt

# Optional: Sort for cleaner plotting (if needed)
  dumbbell_pivot = dumbbell_pivot.sort_values(by='Yes', ascending=False)

# Set up the graph size
  plt.figure(figsize=(8, 5))

# Loop over each row to draw lines and dots
# for idx, row in dumbbell_pivot.iterrows():
  for idx, (i, row) in enumerate(dumbbell_pivot.iterrows()):
# Draw the line (gray line between 'No' and 'Yes')
      plt.plot([row['No'], row['Yes']], [idx, idx], color='gray', linewidth=2, zorder=1)

# Left dot: 'No'
      plt.scatter(row['No'], idx, color='red', s=100, label='No' if idx == 0 else "")

# Right dot: 'Yes'
      plt.scatter(row['Yes'], idx, color='green', s=100, label='Yes' if idx == 0 else "")

# Set y-ticks and label with drug test names
  plt.yticks(ticks=range(len(dumbbell_pivot)), labels=dumbbell_pivot['Urine_Result'])

# Set axis labels and title
  plt.xlabel("Number of Students")
  plt.title("Dumbbell Plot: Yes vs No Urine Test Results by Drug Type")

# Show legend only once
  plt.legend(loc='lower right')

# Optional: Add grid lines for x-axis
  plt.grid(axis='both', linestyle='--', alpha=0.6)

# Tight layout to prevent overlap
  plt.tight_layout()

# Show plot
  plt.show()

# ## BLOCKED


# # Set style and plot THC & COCAINE

#   sns.set(style="darkgrid")

# # Create a figure with 1 row and 2 columns
#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))  # adjust figsize as needed

# # Plot THC
#   sns.barplot(data=thc_pd, x="THC_Positive_Urine", y="count", hue="THC_Positive_Urine",
#                                                   palette="Set1", ax=ax1)
#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax1.set_xlabel("THC_Positive_Urine Values")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of THC_Positive_Urine Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Plot Cocaine
#   sns.barplot(data=cocaine_pd, x="Cocaine_Positive_Urine", y="count", hue="Cocaine_Positive_Urine",
#                                                     palette="Set2", ax=ax2)
#   for p in ax2.patches:
#       ax2.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax2.set_xlabel("Cocaine_Positive_Urine Values")
#   ax2.set_ylabel("Count")
#   ax2.set_title("Distribution of Cocaine_Positive_Urine Values")
#   ax2.tick_params(axis='x', rotation=60)

# # Adjust layout
#   plt.tight_layout()
#   plt.show()

#   print()
#   print()

# ## Plot OPIOID
#   opioid_counts = df_from_csv.groupBy("Opioid_Positive_Urine").count().orderBy("Opioid_Positive_Urine")

#   print("Grouped counts of Opioid_Positive_Urine...")
#   opioid_counts.show()

#   print()
#   print()

# # Step 2: Convert to Pandas DataFrame
#   opioid_pd = opioid_counts.toPandas()

# # Set style and plot Opioid_Positive_Urine

#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))

# # Plot only on ax1
#   sns.barplot(data=opioid_pd, x="Opioid_Positive_Urine", y="count", hue="Opioid_Positive_Urine",
#               palette="Set1", ax=ax1)

#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)

#   ax1.set_xlabel("Opioid_Positive_Urine")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of Opioid_Positive_Urine Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Hide ax2
#   ax2.axis('off')

#   plt.tight_layout()
#   plt.show()
#   print()
#   print()

# ## Group and Plot behavioral patterns

#   Attendance_counts = df_from_csv.groupBy("Attendance").count().orderBy("Attendance")

#   print("Grouped counts of Attendance...")
#   Attendance_counts.show()

# # Step 2: Convert to Pandas DataFrame
#   Attendance_pd = Attendance_counts.toPandas()

# # Group and count Marks_drop

#   Marks_drop_counts = df_from_csv.groupBy("Marks_drop").count().orderBy("Marks_drop")

#   print("Grouped counts of Marks_drop")
#   Marks_drop_counts.show()

# # Convert to Pandas DataFrame

#   Marks_drop_pd = Marks_drop_counts.toPandas()

# # Set style and plot Attendance & Marks_drop

#   sns.set(style="darkgrid")

# # Create a figure with 1 row and 2 columns
#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))  # adjust figsize as needed

# # Plot Attendance
#   sns.barplot(data=Attendance_pd, x="Attendance", y="count", hue="Attendance",
#                                                   palette="Set2", ax=ax1)
#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax1.set_xlabel("Attendance Values - 0->No, 1->Yes")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of Attendance Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Plot Marks_drop
#   sns.barplot(data=Marks_drop_pd, x="Marks_drop", y="count", hue="Marks_drop",
#                                                     palette="Set3", ax=ax2)
#   for p in ax2.patches:
#       ax2.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax2.set_xlabel("Marks_drop Values - 0->No, 1->Yes")
#   ax2.set_ylabel("Count")
#   ax2.set_title("Distribution of Marks_drop Values")
#   ax2.tick_params(axis='x', rotation=60)

# # Adjust layout
#   plt.tight_layout()
#   plt.show()
#   print()
#   print()

# ## Group and Plot Assignment_delays & Frequent_leave

#   Assignment_delays_counts = df_from_csv.groupBy("Assignment_delays").count().orderBy("Assignment_delays")

#   print("Grouped counts of Assignment_delays...")
#   Assignment_delays_counts.show()

# # Step 2: Convert to Pandas DataFrame
#   Assignment_delays_pd = Assignment_delays_counts.toPandas()

# # Group and count Frequent_leave

#   Frequent_leave_counts = df_from_csv.groupBy("Frequent_leave").count().orderBy("Frequent_leave")

#   print("Grouped counts of Frequent_leave")
#   Frequent_leave_counts.show()

# # Convert to Pandas DataFrame

#   Frequent_leave_pd = Frequent_leave_counts.toPandas()

# # Set style and plot Assignment_delays & Frequent_leave

#   sns.set(style="darkgrid")

# # Create a figure with 1 row and 2 columns
#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))  # adjust figsize as needed

# # Plot Assignment_delays
#   sns.barplot(data=Assignment_delays_pd, x="Assignment_delays", y="count", hue="Assignment_delays",
#                                                   palette="Set1", ax=ax1)
#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax1.set_xlabel("Assignment_delays Values - 0->No, 1->Yes")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of Assignment_delays Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Plot Frequent_leave
#   sns.barplot(data=Frequent_leave_pd, x="Frequent_leave", y="count", hue="Frequent_leave",
#                                                     palette="Set2", ax=ax2)
#   for p in ax2.patches:
#       ax2.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax2.set_xlabel("Frequent_leave Values - 0->No, 1->Yes")
#   ax2.set_ylabel("Count")
#   ax2.set_title("Distribution of Frequent_leave Values")
#   ax2.tick_params(axis='x', rotation=60)

# # Adjust layout
#   plt.tight_layout()
#   plt.show()
#   print()
#   print()

# ## Group and Plot Mood_Swings & Peer_Group_Issues

#   Mood_Swings_counts = df_from_csv.groupBy("Mood_Swings").count().orderBy("Mood_Swings")

#   print("Grouped counts of Mood_Swings...")
#   Mood_Swings_counts.show()

# # Step 2: Convert to Pandas DataFrame
#   Mood_Swings_pd = Mood_Swings_counts.toPandas()

# # Group and count Peer_Group_Issues

#   Peer_Group_Issues_counts = df_from_csv.groupBy("Peer_Group_Issues").count().orderBy("Peer_Group_Issues")

#   print("Grouped counts of Peer_Group_Issues")
#   Peer_Group_Issues_counts.show()

# # Convert to Pandas DataFrame

#   Peer_Group_Issues_pd = Peer_Group_Issues_counts.toPandas()

# # Set style and plot Mood_Swings & Peer_Group_Issues

#   sns.set(style="darkgrid")

# # Create a figure with 1 row and 2 columns
#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))  # adjust figsize as needed

# # Plot Mood_Swings
#   sns.barplot(data=Mood_Swings_pd, x="Mood_Swings", y="count", hue="Mood_Swings",
#                                                   palette="Set1", ax=ax1)
#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax1.set_xlabel("Mood_Swings Values - 0->No, 1->Yes")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of Mood_Swings Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Plot Peer_Group_Issues
#   sns.barplot(data=Peer_Group_Issues_pd, x="Peer_Group_Issues", y="count",
#               hue="Peer_Group_Issues", palette="Set2", ax=ax2)
#   for p in ax2.patches:
#       ax2.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax2.set_xlabel("Peer_Group_Issues Values - 0->No, 1->Yes")
#   ax2.set_ylabel("Count")
#   ax2.set_title("Distribution of Peer_Group_Issues Values")
#   ax2.tick_params(axis='x', rotation=60)

# # Adjust layout
#   plt.tight_layout()
#   plt.show()
#   print()
#   print()

# ## Group and Plot Disciplinary_Actions & Counseling_Referral

#   Disciplinary_Actions_counts = df_from_csv.groupBy("Disciplinary_Actions") \
#                                 .count().orderBy("Disciplinary_Actions")

#   print("Grouped counts of Disciplinary_Actions...")
#   Disciplinary_Actions_counts.show()

# # Step 2: Convert to Pandas DataFrame
#   Disciplinary_Actions_pd = Disciplinary_Actions_counts.toPandas()

# # Group and count Counseling_Referral

#   Counseling_Referral_counts = df_from_csv.groupBy("Counseling_Referral") \
#                                 .count().orderBy("Counseling_Referral")

#   print("Grouped counts of Counseling_Referral")
#   Counseling_Referral_counts.show()

# # Convert to Pandas DataFrame

#   Counseling_Referral_pd = Counseling_Referral_counts.toPandas()

# # Set style and plot Mood_Swings & Peer_Group_Issues

#   sns.set(style="darkgrid")

# # Create a figure with 1 row and 2 columns
#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))  # adjust figsize as needed

# # Plot Disciplinary_Actions
#   sns.barplot(data=Disciplinary_Actions_pd, x="Disciplinary_Actions", y="count",
#                   hue="Disciplinary_Actions", palette="Set1", ax=ax1)
#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax1.set_xlabel("Disciplinary_Actions Values - 0->No, 1->Yes")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of Disciplinary_Actions Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Plot Counseling_Referral
#   sns.barplot(data=Counseling_Referral_pd, x="Counseling_Referral", y="count",
#               hue="Counseling_Referral", palette="Set2", ax=ax2)
#   for p in ax2.patches:
#       ax2.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax2.set_xlabel("Counseling_Referral Values - 0->No, 1->Yes")
#   ax2.set_ylabel("Count")
#   ax2.set_title("Distribution of Counseling_Referral Values")
#   ax2.tick_params(axis='x', rotation=60)

# # Adjust layout
#   plt.tight_layout()
#   plt.show()
#   print()
#   print()

# ## Group and Plot Social_Media_Overuse & Gaming_Addiction

#   Social_Media_Overuse_counts = df_from_csv.groupBy("Social_Media_Overuse") \
#                                 .count().orderBy("Social_Media_Overuse")

#   print("Grouped counts of Social_Media_Overuse...")
#   Social_Media_Overuse_counts.show()

# # Step 2: Convert to Pandas DataFrame
#   Social_Media_Overuse_pd = Social_Media_Overuse_counts.toPandas()

# # Group and count Gaming_Addiction

#   Gaming_Addiction_counts = df_from_csv.groupBy("Gaming_Addiction") \
#                                 .count().orderBy("Gaming_Addiction")

#   print("Grouped counts of Gaming_Addiction")
#   Gaming_Addiction_counts.show()

# # Convert to Pandas DataFrame

#   Gaming_Addiction_pd = Gaming_Addiction_counts.toPandas()

# # Set style and plot Social_Media_Overuse & Gaming_Addiction

#   sns.set(style="darkgrid")

# # Create a figure with 1 row and 2 columns
#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))  # adjust figsize as needed

# # Plot Disciplinary_Actions
#   sns.barplot(data=Social_Media_Overuse_pd, x="Social_Media_Overuse", y="count",
#                   hue="Social_Media_Overuse", palette="Set1", ax=ax1)
#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax1.set_xlabel("Social_Media_Overuse Values - 0->No, 1->Yes")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of Social_Media_Overuse Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Plot Gaming_Addiction
#   sns.barplot(data=Gaming_Addiction_pd, x="Gaming_Addiction", y="count",
#               hue="Gaming_Addiction", palette="Set2", ax=ax2)
#   for p in ax2.patches:
#       ax2.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)
#   ax2.set_xlabel("Gaming_Addiction Values - 0->No, 1->Yes")
#   ax2.set_ylabel("Count")
#   ax2.set_title("Distribution of Gaming_Addiction Values")
#   ax2.tick_params(axis='x', rotation=60)

# # Adjust layout
#   plt.tight_layout()
#   plt.show()
#   print()
#   print()

# ## Plot Distribution of Addiction_Label only

#   Addiction_Label_counts = df_from_csv.groupBy("Addiction_Label") \
#                               .count().orderBy("Addiction_Label")

#   print("Grouped counts of Addiction_Label...")
#   Addiction_Label_counts.show()

#   print()
#   print()

# # Step 2: Convert to Pandas DataFrame
#   Addiction_Label_pd = Addiction_Label_counts.toPandas()

# # Set style and plot Addiction_Label

#   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))

# # Plot only on ax1
#   sns.barplot(data=Addiction_Label_pd, x="Addiction_Label", y="count", \
#               hue="Addiction_Label", palette="Set1", ax=ax1)

#   for p in ax1.patches:
#       ax1.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
#                   ha='center', va='bottom', fontsize=9)

#   ax1.set_xlabel("Addiction_Label")
#   ax1.set_ylabel("Count")
#   ax1.set_title("Distribution of Addiction_Label Values")
#   ax1.tick_params(axis='x', rotation=60)

# # Hide ax2
#   ax2.axis('off')

#   plt.tight_layout()
#   plt.show()
#   print()
#   print()
# ## BLOCK ENDS HERE
  print()
  display(HTML("<h3 style='color:orange; text-align:left;'>EDA Finished !!!</h3>"))

  go_back_function(event)

def pre_process_data(event):

  print()
  print()
  clear_result_grid()

  display(HTML("<h3 style='color:orange; text-align:left;'>Pre-Processing of Data Started, Please wait !!!</h3>"))
  print()

## df_from_csv_new is without any Duplicates

  global df_balanced_indexed

## Take count of null values in each col
## .alias(c) to get original column name in the dataset

  null_counts = df_from_csv.select([
      _sum(col(c).isNull().cast("int")).alias(c) for c in df_from_csv.columns
  ])

  print("Null Values count in each column...")
  null_counts.show()

# Null Values count in each column...
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+
# |StudentID|ALT_Blood|GGT_Blood|Glucose_Blood|Creatine_Blood|THC_Positive_Urine|Cocaine_Positive_Urine|Opioid_Positive_Urine|Urine_pH|Attendance|Marks_drop|Assignment_delays|Frequent_leave|Mood_Swings|Peer_Group_Issues|Disciplinary_Actions|Counseling_Referral|Social_Media_Overuse|Gaming_Addiction|Addiction_Label|
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+
# |        0|        0|        0|            0|             0|                 0|                     0|                    0|       0|         0|         0|                0|             0|          0|                0|                   0|                  0|                   0|               0|              0|
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+

## Remove all NULL value rows. Since there is no null or missing values the following
## .drop is not needed

  # df_no_nulls = df_from_csv.na.drop()

## Count Duplicate Rows

  total_rows = df_from_csv.count()
  distinct_rows = df_from_csv.distinct().count()
  duplicate_count = total_rows - distinct_rows

  print(f"Total rows: {total_rows}")
  print(f"Total duplicate rows: {duplicate_count}")

# Total rows: 75000
# Total duplicate rows: 8067

# ## Remove Duplicates - it may affect model efficiency

  df_from_csv_new = df_from_csv.dropDuplicates()

## CROSS VERIFY
  total_rows_after = df_from_csv_new.count()
  print(f"Total rows after removing Duplicates: {total_rows_after}")

# Total rows after removing Duplicates: 66933

## Verify Addiction_Label after removing Duplicates
  Addiction_Label_counts = df_from_csv_new.groupBy("Addiction_Label") \
                              .count().orderBy("Addiction_Label")

  print("Grouped counts of Addiction_Label after duplicates removed...")
  Addiction_Label_counts.show()

# Grouped counts of Addiction_Label after duplicates removed...
# +---------------+-----+
# |Addiction_Label|count|
# +---------------+-----+
# |       Addicted|28495|  With Duplicates-->31986
# |   Not Addicted|38438|  With Duplicates-->43014
# +---------------+-----+

# ## CROSS VERIFY SCHEMA of df_from_csv_new

  print("Display schema of df_from_csv_new")
  df_from_csv_new.printSchema()
# Display schema of df_from_csv_new
# root
#  |-- StudentID: string (nullable = true)
#  |-- ALT_Blood: integer (nullable = true)
#  |-- GGT_Blood: integer (nullable = true)
#  |-- Glucose_Blood: integer (nullable = true)
#  |-- Creatine_Blood: double (nullable = true)
#  |-- THC_Positive_Urine: string (nullable = true)
#  |-- Cocaine_Positive_Urine: string (nullable = true)
#  |-- Opioid_Positive_Urine: string (nullable = true)
#  |-- Urine_pH: double (nullable = true)
#  |-- Attendance: integer (nullable = true)
#  |-- Marks_drop: integer (nullable = true)
#  |-- Assignment_delays: integer (nullable = true)
#  |-- Frequent_leave: integer (nullable = true)
#  |-- Mood_Swings: integer (nullable = true)
#  |-- Peer_Group_Issues: integer (nullable = true)
#  |-- Disciplinary_Actions: integer (nullable = true)
#  |-- Counseling_Referral: integer (nullable = true)
#  |-- Social_Media_Overuse: integer (nullable = true)
#  |-- Gaming_Addiction: integer (nullable = true)
#  |-- Addiction_Label: string (nullable = true)

# ## Clean 'Addiction_Label' col if it contains abnormal values

  chk_Addiction_Label = df_from_csv_new.groupBy("Addiction_Label").count() \
                        .orderBy("Addiction_Label")
  print("chk_Addiction_Label")
  chk_Addiction_Label.show(20)
# chk_Addiction_Label
# +---------------+-----+
# |Addiction_Label|count|
# +---------------+-----+
# |       Addicted|28495|  This col is clean only 2 possibilities found
# |   Not Addicted|38438|
# +---------------+-----+

# ######### Point-Biserial correlation Analysis starts here

  string_cols = ['THC_Positive_Urine', 'Cocaine_Positive_Urine',
                 'Opioid_Positive_Urine', 'Addiction_Label']

# # Apply String indexer to convert string to numeric

# Create StringIndexers with alphabetic order

  indexers = [
          StringIndexer(
          inputCol=col,
          outputCol=f"{col}_indexed",
          handleInvalid='skip',
          stringOrderType='alphabetAsc'  # 👈 forces alphabetical indexing
          )
          for col in string_cols
          ]

# # Create and apply pipeline - Safe to ignore the following warning
  pipeline = Pipeline(stages=indexers)

  pipeline_model_indexed = pipeline.fit(df_from_csv_new)

  df_balanced_indexed= pipeline_model_indexed.transform(df_from_csv_new)

  print("Print Schema of df_balanced_indexed...")
  df_balanced_indexed.printSchema()

# Print Schema of df_balanced_indexed...
# root
#  |-- StudentID: string (nullable = true)
#  |-- ALT_Blood: integer (nullable = true)
#  |-- GGT_Blood: integer (nullable = true)
#  |-- Glucose_Blood: integer (nullable = true)
#  |-- Creatine_Blood: double (nullable = true)
#  |-- THC_Positive_Urine: string (nullable = true)
#  |-- Cocaine_Positive_Urine: string (nullable = true)
#  |-- Opioid_Positive_Urine: string (nullable = true)
#  |-- Urine_pH: double (nullable = true)
#  |-- Attendance: integer (nullable = true)
#  |-- Marks_drop: integer (nullable = true)
#  |-- Assignment_delays: integer (nullable = true)
#  |-- Frequent_leave: integer (nullable = true)
#  |-- Mood_Swings: integer (nullable = true)
#  |-- Peer_Group_Issues: integer (nullable = true)
#  |-- Disciplinary_Actions: integer (nullable = true)
#  |-- Counseling_Referral: integer (nullable = true)
#  |-- Social_Media_Overuse: integer (nullable = true)
#  |-- Gaming_Addiction: integer (nullable = true)
#  |-- Addiction_Label: string (nullable = true)
#  |-- THC_Positive_Urine_indexed: double (nullable = false)
#  |-- Cocaine_Positive_Urine_indexed: double (nullable = false)
#  |-- Opioid_Positive_Urine_indexed: double (nullable = false)
#  |-- Addiction_Label_indexed: double (nullable = false)

  print("Display indexed df_balanced_indexed...")
  df_balanced_indexed.show(5, False)

# Display indexed df_balanced_indexed... No-->0, Yes-->1, Addicted-->0, Not Addicted-->1
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+
# |StudentID|ALT_Blood|GGT_Blood|Glucose_Blood|Creatine_Blood|THC_Positive_Urine|Cocaine_Positive_Urine|Opioid_Positive_Urine|Urine_pH|Attendance|Marks_drop|Assignment_delays|Frequent_leave|Mood_Swings|Peer_Group_Issues|Disciplinary_Actions|Counseling_Referral|Social_Media_Overuse|Gaming_Addiction|Addiction_Label|THC_Positive_Urine_indexed|Cocaine_Positive_Urine_indexed|Opioid_Positive_Urine_indexed|Addiction_Label_indexed|
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+
# |   S01101|       74|       17|           70|          1.26|                No|                    No|                   No|     6.1|         0|         0|                1|             0|          1|                1|                   0|                  0|                   1|               0|   Not Addicted|                       0.0|                           0.0|                          0.0|                    1.0|
# |   S00878|       58|      112|          128|           1.4|                No|                    No|                  Yes|     5.8|         0|         0|                0|             0|          1|                1|                   1|                  1|                   1|               1|       Addicted|                       0.0|                           0.0|                          1.0|                    0.0|
# |   S01685|       66|      158|          147|          1.45|                No|                    No|                  Yes|     6.1|         0|         0|                1|             1|          0|                1|                   1|                  1|                   0|               0|       Addicted|                       0.0|                           0.0|                          1.0|                    0.0|
# |   S01411|       54|       99|           80|          1.08|                No|                    No|                   No|     6.5|         0|         0|                0|             0|          0|                0|                   0|                  0|                   0|               0|   Not Addicted|                       0.0|                           0.0|                          0.0|                    1.0|
# |   S02132|       59|      118|          136|          1.12|                No|                    No|                   No|     5.9|         0|         0|                1|             0|          1|                0|                   0|                  1|                   0|               0|   Not Addicted|                       0.0|                           0.0|                          0.0|                    1.0|
# |   S04080|       63|       92|          110|          1.22|               Yes|                    No|                   No|     5.9|         0|         1|                0|             0|          0|                0|                   1|                  0|                   0|               0|   Not Addicted|                       1.0|                           0.0|                          0.0|                    1.0|
# |   S02121|       60|       63|          109|          0.76|               Yes|                    No|                   No|     5.8|         0|         1|                0|             1|          0|                1|                   0|                  0|                   1|               1|   Not Addicted|                       1.0|                           0.0|                          0.0|                    1.0|

  from pyspark.sql.functions import isnan
  print("Check any null values...")
  null_in_target= df_balanced_indexed.filter(isnan("Addiction_Label_indexed")).count()
  print(null_in_target)
## Save the above indexed values for future use

  pipeline_model_indexed.write().overwrite() \
            .save("/content/Student-Drug-Addiction-Prediction/cols_indexed_drug_addiction")

## Filter only relevant cols from df_balanced_indexed

  df_balanced_filtered = df_balanced_indexed.select(
                  "ALT_Blood","GGT_Blood","Glucose_Blood",
                  "Creatine_Blood","THC_Positive_Urine_indexed",
                  "Cocaine_Positive_Urine_indexed",
                  "Opioid_Positive_Urine_indexed","Urine_pH",
                  "Attendance","Marks_drop",
                  "Assignment_delays","Frequent_leave",
                  "Mood_Swings","Peer_Group_Issues",
                  "Disciplinary_Actions","Counseling_Referral",
                  "Social_Media_Overuse","Gaming_Addiction",
				          "Addiction_Label_indexed"
						)

# Convert df_balanced_filtered to pandas

  df_balanced_filtered_pd = df_balanced_filtered.toPandas()

  target = 'Addiction_Label_indexed'
  features_corr = [cols for cols in df_balanced_filtered_pd.columns if cols != target]

# Compute point-biserial correlation

  from scipy.stats import pointbiserialr

#   # Store results from the correlation computation

  correlation_results = []
  selected_features = []

  for feature in features_corr:
      r, p = pointbiserialr(df_balanced_filtered_pd['Addiction_Label_indexed'],
                            df_balanced_filtered_pd[feature])
      print(f"{feature:35}  r = {r:.4f},  p-value = {p:.4g}")
      correlation_results.append({'Feature': feature, 'r_value': r, 'p_value': p})

# Apply relaxed selection criteria
      if abs(r) >= 0.1 and p < 0.1:
          selected_features.append(feature)

# Print selected features
  print("📌 Selected Features with Slight or Strong Relationship to Addiction_Label_indexed:\n")
  for feat in selected_features:
      print(f"✅ {feat}")


# ## Display bar chart

# Convert to DataFrame
  df_corr = pd.DataFrame(correlation_results)

# Handle potential NaNs (e.g., constant columns) by filling them with 0
  df_corr['r_value'] = df_corr['r_value'].fillna(0)
  df_corr['abs_r'] = df_corr['r_value'].abs()

# Sort by absolute correlation for visual impact
  df_corr_sorted = df_corr.sort_values(by='abs_r', ascending=False)

# Plotting the bar chart
  plt.figure(figsize=(12, 7))
  plt.barh(df_corr_sorted['Feature'], df_corr_sorted['r_value'], color='skyblue')
  plt.xlabel('Point-Biserial Correlation (r)', fontsize=12)
  plt.title('Feature Correlation with Addiction_Label_indexed', fontsize=14)
  plt.gca().invert_yaxis()
  plt.grid(axis='x', linestyle='--', alpha=0.6)
  plt.tight_layout()
  plt.show()

## Co-Relation output --> Ignore r = 0

# ALT_Blood                            r = -0.1916,  p-value = 0
# GGT_Blood                            r = -0.2082,  p-value = 0
# Glucose_Blood                        r = -0.0042,  p-value = 0.2829
# Creatine_Blood                       r = -0.1377,  p-value = 1.339e-280
# THC_Positive_Urine_indexed           r = -0.2111,  p-value = 0
# Cocaine_Positive_Urine_indexed       r = -0.1392,  p-value = 1.184e-286
# Opioid_Positive_Urine_indexed        r = -0.1730,  p-value = 0
# Urine_pH                             r = -0.0012,  p-value = 0.7511
# Attendance                           r = 0.0102,   p-value = 0.008125
# Marks_drop                           r = 0.0063,   p-value = 0.1027
# Assignment_delays                    r = -0.0161,  p-value = 3.022e-05
# Frequent_leave                       r = -0.0034,  p-value = 0.3824
# Mood_Swings                          r = -0.1983,  p-value = 0
# Peer_Group_Issues                    r = -0.2065,  p-value = 0
# Disciplinary_Actions                 r = -0.2053,  p-value = 0
# Counseling_Referral                  r = -0.2014,  p-value = 0
# Social_Media_Overuse                 r = -0.1980,  p-value = 0
# Gaming_Addiction                     r = -0.2091,  p-value = 0

# ### ✅ Summary of Feature Selection:

# 🔴 High Importance

# THC_Positive_Urine_indexed
# Cocaine_Positive_Urine_indexed
# Opioid_Positive_Urine_indexed
# Gaming_Addiction
# Social_Media_Overuse
# Peer_Group_Issues
# Mood_Swings
# Counseling_Referral
# Disciplinary_Actions

# 🟠 Medium Importance - Consider these

# ALT_Blood, GGT_Blood, Creatine_Blood (biomarkers, moderate correlation)
# Assignment_delays (behavioral indicator)
# Attendance
# Frequent_leave and Marks_drop

# 🟢 Low or Insignificant Ignore

# Urine_pH and Glucose_Blood – not significant (high p-values)

  print()
  print()
  display(HTML("<h3 style='color:orange; text-align:left;'>Pre-Processing Finished !!!</h3>"))
  go_back_function(event)

def transformation(event):
  print()
  print()
  clear_result_grid()

  display(HTML("<h3 style='color:orange; text-align:left;'>Transformation \
                          of Data Started, Please wait !!!</h3>"))

  global df_vectorized_lr, df_vectorized_rf

## Consider the following for Logistic Regression Features as per the
## correlation analysis, 'Glucose_Blood' and 'Urine_pH' is removed

  lr_features = ['ALT_Blood','GGT_Blood',
                'Creatine_Blood','Attendance',
                'Marks_drop', 'Assignment_delays','Frequent_leave',
                'Mood_Swings','Peer_Group_Issues',
                'Disciplinary_Actions','Counseling_Referral',
                'Social_Media_Overuse', 'Gaming_Addiction',
                'THC_Positive_Urine_indexed',
                'Cocaine_Positive_Urine_indexed',
                'Opioid_Positive_Urine_indexed'

                ]

# Define the output column name for the assembled feature vectors

  output_col_lr_vector = "features_vector_lr"

# Vectorize the above feature_cols and add it as a list in df_transformed_lr

  assembler_lr = VectorAssembler(inputCols=lr_features, outputCol=output_col_lr_vector)

  df_vectorized_lr = assembler_lr.transform(df_balanced_indexed)

## CROSS VERIFICATION

  print("Display Schema of df_vectorized_lr .....")
  df_vectorized_lr.printSchema()

  print("Display data in df_vectorized_lr...")
  df_vectorized_lr.show(5, False)

## Vector Explanation
## Sparse and Dense Vector distribution is managed by pyspark library
# +--------------------------------------------------------------------+
# |features_vector_lr                                                  |
# +--------------------------------------------------------------------+
# |(16,[0,1,2,5,7,8,11],[74.0,17.0,1.26,1.0,1.0,1.0,1.0])              |
# |[58.0,112.0,1.4,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0]|
# |(16,[0,1,2,5,6,8,9,10,15],[66.0,158.0,1.45,1.0,1.0,1.0,1.0,1.0,1.0])|
# |(16,[0,1,2],[54.0,99.0,1.08])                                       |
# |(16,[0,1,2,5,7,10],[59.0,118.0,1.12,1.0,1.0,1.0])                   |
# +--------------------------------------------------------------------+

## The above data segment shows features after vectorized
## The output is divided into 'SPARSE' AND 'DENSE' VECTORS

## 'SPARSE' shows (size, indices, values)
## (16,[....]), 16 denotes total feature cols

## [0,1,2,5,7,8,11] Indices of list and values which contains Non Zero Values only
## [74.0,17.0,1.26,1.0,1.0,1.0,1.0]), shows actual values at each index
## That is ALT_Blood at index 0 contains a decimal value 74.0
## CGT_Blood at index 1 contains a decimal value 17.0 and so on

## DENSE Vectors - second row shows all values

# [58.0,112.0,1.4,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0]

# Display data in df_vectorized_lr...
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+--------------------------------------------------------------------+
# |StudentID|ALT_Blood|GGT_Blood|Glucose_Blood|Creatine_Blood|THC_Positive_Urine|Cocaine_Positive_Urine|Opioid_Positive_Urine|Urine_pH|Attendance|Marks_drop|Assignment_delays|Frequent_leave|Mood_Swings|Peer_Group_Issues|Disciplinary_Actions|Counseling_Referral|Social_Media_Overuse|Gaming_Addiction|Addiction_Label|THC_Positive_Urine_indexed|Cocaine_Positive_Urine_indexed|Opioid_Positive_Urine_indexed|Addiction_Label_indexed|features_vector_lr                                                  |
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+--------------------------------------------------------------------+
# |S01101   |74       |17       |70           |1.26          |No                |No                    |No                   |6.1     |0         |0         |1                |0             |1          |1                |0                   |0                  |1                   |0               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(16,[0,1,2,5,7,8,11],[74.0,17.0,1.26,1.0,1.0,1.0,1.0])              |
# |S00878   |58       |112      |128          |1.4           |No                |No                    |Yes                  |5.8     |0         |0         |0                |0             |1          |1                |1                   |1                  |1                   |1               |Addicted       |0.0                       |0.0                           |1.0                          |0.0                    |[58.0,112.0,1.4,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0]|
# |S01685   |66       |158      |147          |1.45          |No                |No                    |Yes                  |6.1     |0         |0         |1                |1             |0          |1                |1                   |1                  |0                   |0               |Addicted       |0.0                       |0.0                           |1.0                          |0.0                    |(16,[0,1,2,5,6,8,9,10,15],[66.0,158.0,1.45,1.0,1.0,1.0,1.0,1.0,1.0])|
# |S01411   |54       |99       |80           |1.08          |No                |No                    |No                   |6.5     |0         |0         |0                |0             |0          |0                |0                   |0                  |0                   |0               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(16,[0,1,2],[54.0,99.0,1.08])                                       |
# |S02132   |59       |118      |136          |1.12          |No                |No                    |No                   |5.9     |0         |0         |1                |0             |1          |0                |0                   |1                  |0                   |0               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(16,[0,1,2,5,7,10],[59.0,118.0,1.12,1.0,1.0,1.0])                   |
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+--------------------------------------------------------------------+
# only showing top 5 rows

# ## Similarly Vectorize Tree features - here all features are included because
# ## Trees can identify weak patterns in Glucose_Blood, and Urine_pH

  rf_features = ['ALT_Blood','GGT_Blood','Glucose_Blood',
                'Creatine_Blood','Urine_pH','Attendance',
                'Marks_drop', 'Assignment_delays','Frequent_leave',
                'Mood_Swings','Peer_Group_Issues',
                'Disciplinary_Actions','Counseling_Referral',
                'Social_Media_Overuse', 'Gaming_Addiction',
                'THC_Positive_Urine_indexed',
                'Cocaine_Positive_Urine_indexed',
                'Opioid_Positive_Urine_indexed'
                ]

  output_col_rf_vector = "features_vector_rf"

# Vectorize the above feature_cols and add it as a list in df_transformed_rf

  assembler_rf = VectorAssembler(inputCols=rf_features, outputCol=output_col_rf_vector)

  df_vectorized_rf = assembler_rf.transform(df_balanced_indexed)

# ## CROSS VERIFICATION

  print("Display Schema of df_vectorized_rf .....")
  df_vectorized_rf.printSchema()

  print("Display data in df_vectorized_rf...")
  df_vectorized_rf.show(5, False)
# Display data in df_vectorized_rf...
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+-------------------------------------------------------------------------------+
# |StudentID|ALT_Blood|GGT_Blood|Glucose_Blood|Creatine_Blood|THC_Positive_Urine|Cocaine_Positive_Urine|Opioid_Positive_Urine|Urine_pH|Attendance|Marks_drop|Assignment_delays|Frequent_leave|Mood_Swings|Peer_Group_Issues|Disciplinary_Actions|Counseling_Referral|Social_Media_Overuse|Gaming_Addiction|Addiction_Label|THC_Positive_Urine_indexed|Cocaine_Positive_Urine_indexed|Opioid_Positive_Urine_indexed|Addiction_Label_indexed|features_vector_rf                                                             |
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+-------------------------------------------------------------------------------+
# |S01101   |74       |17       |70           |1.26          |No                |No                    |No                   |6.1     |0         |0         |1                |0             |1          |1                |0                   |0                  |1                   |0               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(18,[0,1,2,3,4,7,9,10,13],[74.0,17.0,70.0,1.26,6.1,1.0,1.0,1.0,1.0])           |
# |S00878   |58       |112      |128          |1.4           |No                |No                    |Yes                  |5.8     |0         |0         |0                |0             |1          |1                |1                   |1                  |1                   |1               |Addicted       |0.0                       |0.0                           |1.0                          |0.0                    |[58.0,112.0,128.0,1.4,5.8,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0] |
# |S01685   |66       |158      |147          |1.45          |No                |No                    |Yes                  |6.1     |0         |0         |1                |1             |0          |1                |1                   |1                  |0                   |0               |Addicted       |0.0                       |0.0                           |1.0                          |0.0                    |[66.0,158.0,147.0,1.45,6.1,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0]|
# |S01411   |54       |99       |80           |1.08          |No                |No                    |No                   |6.5     |0         |0         |0                |0             |0          |0                |0                   |0                  |0                   |0               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(18,[0,1,2,3,4],[54.0,99.0,80.0,1.08,6.5])                                     |
# |S02132   |59       |118      |136          |1.12          |No                |No                    |No                   |5.9     |0         |0         |1                |0             |1          |0                |0                   |1                  |0                   |0               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(18,[0,1,2,3,4,7,9,12],[59.0,118.0,136.0,1.12,5.9,1.0,1.0,1.0])                |
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+-------------------------------------------------------------------------------+
# only showing top 5 rows

  print()
  display(HTML("<h3 style='color:orange; text-align:left;'>Transformation \
                          Finished !!!</h3>"))
  go_back_function(event)

def build_model_save(event):
  print()
  clear_result_grid()
  set_focus_cell2()

  # display(HTML("<h3 style='color:magenta; text-align:left;'> \
  #   Model Building Started Please Wait...will take 1 to 2 minutes..</h3>"))

## Split Train and Test Data

  train_ratio = 0.70
  test_ratio = 0.30

  train_data_lr, test_data_lr = df_vectorized_lr \
                                .randomSplit([train_ratio, test_ratio],
                                 seed=5043)

  display(HTML("<h3 style='color:magenta; text-align:left;'> \
    Logistic Regression Started Please Wait...will take 1 to 2 minutes..</h3>"))

  print("train_data_lr verification")
  train_data_lr.select("Addiction_Label_indexed").distinct().show()

## CROSS VERIFICATION

# Count total number of rows in train_data_lr, test_data_lr

  tot_train_lr = train_data_lr.count()
  print("Total Rows in Training Dataset..LR=", tot_train_lr)

  tot_test_lr = test_data_lr.count()
  print("Total Rows in Test Dataset..LR=", tot_test_lr)

  print("Check target var class distribution in train_data_lr...")
  train_data_lr.select("Addiction_Label_indexed").distinct().show()

  print("Check target var class distribution in test_data_lr...")
  test_data_lr.select("Addiction_Label_indexed").distinct().show()

## Create the Model - Detailed Code explanation as follows:-

# featuresCol="features_vector_lr"
# This is the name of the input column that contains your
# feature vector (i.e., all your input features combined into Spare or Dense vector
# using VectorAssembler).
# features_vector_lr should be a column of type Vector.

# labelCol="Addiction_Label_indexed"
# This is the name of the column that contains the target labels.

# In your case, this column likely contains binary values
# (e.g., 0 for Addicted, 1 for Not Addicted)
# possibly created using StringIndexer.

# maxIter=100
# This sets the maximum number of iterations the optimizer will run to find the
# best model coefficients.
# A higher value allows more time to converge but can increase training time.

# regParam=0.01
# This is the regularization parameter (λ).

# It helps prevent overfitting by penalizing large coefficients. A small value
# like 0.01 means light regularization.

### The Regularization parameters are part of Mathematics[Linear Algebra] and
###                                           Statistics

# elasticNetParam=0.5
# This controls the type of regularization:

# 0.0 = L2 regularization (Ridge)
# 1.0 = L1 regularization (Lasso)
# 0.5 = a mix of both (Elastic Net)
# So, 0.5 means Elastic Net regularization, balancing between L1 and L2.

# family="binomial"
# This tells the model that you're solving a binary classification problem
# (i.e., 2 classes--> 0 or 1).

# If you had more than 2 classes, you'd use "multinomial" instead.

  lr = LogisticRegression(
      featuresCol="features_vector_lr",
      labelCol="Addiction_Label_indexed",
      maxIter=100,
      regParam=0.01,
      elasticNetParam=0.5,
      family="binomial"
  )

## Train the model using dataframe train_data_lr

  lr_model = lr.fit(train_data_lr)

## Make Predictions using test_data_lr

  predictions_lr = lr_model.transform(test_data_lr)

  print("Display Schema of predictions_lr .....")
  predictions_lr.printSchema()

  print("Few data in predictions_lr....")
  predictions_lr.show(5, False)
# Few data in predictions_lr....
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+---------------------------------------------------------------------+------------------------------------------+-----------------------------------------+----------+
# |StudentID|ALT_Blood|GGT_Blood|Glucose_Blood|Creatine_Blood|THC_Positive_Urine|Cocaine_Positive_Urine|Opioid_Positive_Urine|Urine_pH|Attendance|Marks_drop|Assignment_delays|Frequent_leave|Mood_Swings|Peer_Group_Issues|Disciplinary_Actions|Counseling_Referral|Social_Media_Overuse|Gaming_Addiction|Addiction_Label|THC_Positive_Urine_indexed|Cocaine_Positive_Urine_indexed|Opioid_Positive_Urine_indexed|Addiction_Label_indexed|features_vector_lr                                                   |rawPrediction                             |probability                              |prediction|
# +---------+---------+---------+-------------+--------------+------------------+----------------------+---------------------+--------+----------+----------+-----------------+--------------+-----------+-----------------+--------------------+-------------------+--------------------+----------------+---------------+--------------------------+------------------------------+-----------------------------+-----------------------+---------------------------------------------------------------------+------------------------------------------+-----------------------------------------+----------+
# |S00101   |56       |121      |108          |0.87          |No                |No                    |Yes                  |7.0     |1         |1         |0                |0             |0          |1                |1                   |1                  |1                   |1               |Addicted       |0.0                       |0.0                           |1.0                          |0.0                    |[56.0,121.0,0.87,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0]|[2.5968460770960142,-2.5968460770960142]  |[0.9306583225043092,0.06934167749569076] |0.0       |
# |S00102   |45       |148      |112          |1.28          |No                |No                    |No                   |5.8     |0         |1         |0                |0             |1          |1                |0                   |0                  |0                   |1               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(16,[0,1,2,4,7,8,12],[45.0,148.0,1.28,1.0,1.0,1.0,1.0])              |[-0.4881249169124615,0.4881249169124615]  |[0.3803353886872763,0.6196646113127238]  |1.0       |
# |S00103   |73       |94       |155          |0.75          |No                |No                    |No                   |6.3     |0         |0         |0                |0             |0          |1                |1                   |1                  |0                   |1               |Addicted       |0.0                       |0.0                           |0.0                          |0.0                    |(16,[0,1,2,8,9,10,12],[73.0,94.0,0.75,1.0,1.0,1.0,1.0])              |[-0.07718864584369278,0.07718864584369278]|[0.48071241401461123,0.5192875859853887] |1.0       |
# |S00104   |10       |71       |75           |0.86          |No                |No                    |No                   |6.3     |0         |1         |0                |0             |0          |0                |0                   |0                  |1                   |1               |Not Addicted   |0.0                       |0.0                           |0.0                          |1.0                    |(16,[0,1,2,4,11,12],[10.0,71.0,0.86,1.0,1.0,1.0])                    |[-4.312301847978992,4.312301847978992]    |[0.013225407485062676,0.9867745925149374]|1.0       |
# |S00105   |42       |46       |145          |0.9           |Yes               |No                    |No                   |6.0     |0         |0         |1                |0             |1          |0                |1                   |0                  |1                   |0               |Not Addicted   |1.0                       |0.0                           |0.0                          |1.0                    |(16,[0,1,2,5,7,9,11,13],[42.0,46.0,0.9,1.0,1.0,1.0,1.0,1.0])         |[-1.1971880954274514,1.1971880954274514]  |[0.2319758163481801,0.7680241836518199]  |1.0       |

## Evaluate the Model - Check Accuracy,
## You can use a binary evaluator (since the target is 0 or 1):

  evaluator_lr = BinaryClassificationEvaluator(
      labelCol="Addiction_Label_indexed",
      rawPredictionCol="rawPrediction",
      metricName="areaUnderROC"  # or "areaUnderPR"
      )

  auc = evaluator_lr.evaluate(predictions_lr)
  print(f"Accuracy using AUC = {auc}")

# Accuracy using AUC = 0.9110372842721056

# Confusion Matrix
# To compute the confusion matrix manually:

  predictions_lr.groupBy("Addiction_Label_indexed", "prediction").count().show()


  display(HTML("<h3 style='color:green; text-align:left;'> \
    Random Forest Started Please Wait...will take 5 to 8 minutes..</h3>"))

# # Create RandomForest model

  train_data_rf, test_data_rf = df_vectorized_rf \
                                .randomSplit([train_ratio, test_ratio],
                                 seed=5043)

  train_data_rf.cache()
  test_data_rf.cache()

# Increasing number of trees and depth can cause 'overfitting'
# numTrees=20 and maxDepth=10 is stable combination.

  rf = RandomForestClassifier(
      featuresCol="features_vector_rf",  # same features column as LR
      labelCol="Addiction_Label_indexed",      # same label column
      numTrees=20,                      # similar to maxIter, controls number of trees
      maxDepth=7                         # optional: controls depth of each tree
  )

# Train the model
  rf_model = rf.fit(train_data_rf)

# Make predictions
  predictions_rf = rf_model.transform(test_data_rf)

# 1. Accuracy

  evaluator_acc = MulticlassClassificationEvaluator(
      labelCol="Addiction_Label_indexed",
      predictionCol="prediction",
      metricName="accuracy"
  )

  accuracy = evaluator_acc.evaluate(predictions_rf)
  print(f"Accuracy = {accuracy:.4f}")

# 2. F1 Score
  evaluator_f1 = MulticlassClassificationEvaluator(
      labelCol="Addiction_Label_indexed",
      predictionCol="prediction",
      metricName="f1"
  )
  f1 = evaluator_f1.evaluate(predictions_rf)
  print(f"F1 Score = {f1:.4f}")

# 3. Precision & Recall using MulticlassMetrics

  evaluator_precision = MulticlassClassificationEvaluator(
      labelCol="Addiction_Label_indexed",
      predictionCol="prediction",
      metricName="weightedPrecision"
  )

  precision = evaluator_precision.evaluate(predictions_rf)
  print("Precision:", precision)

# ## OUTPUT - in percent, multiply by 100

# Accuracy = 0.9504
# F1 Score = 0.9501
# Precision: 0.9525

## Plot Confusion Matrix Heatmap

# Step 1: Create confusion matrix using pivot table
  confusion_df = predictions_rf.groupBy("Addiction_Label_indexed") \
      .pivot("prediction") \
      .count() \
      .na.fill(0) \
      .orderBy("Addiction_Label_indexed")

# Step 2: Collect to Pandas (small data only!)
  confusion_pd = confusion_df.toPandas()

# Step 3: Set index to actual labels
  confusion_pd.set_index("Addiction_Label_indexed", inplace=True)

# Step 4: Plot heatmap
  plt.figure(figsize=(8,6))
  sns.heatmap(confusion_pd, annot=True, fmt="d", cmap="Blues", cbar=True)

  plt.title("Confusion Matrix Heatmap")
  plt.xlabel("Predicted Label")
  plt.ylabel("Actual Label")
  plt.tight_layout()
  plt.show()

## CATBOOST starting here

  display(HTML("<h3 style='color:red; text-align:left;'> \
    CATBOOST Algorithm Started Please Wait...will take 1 to 2 minutes..</h3>"))

## Convert pyspark dataframe
## Remove string indexed cols in df_balanced_indexed

  cat_df = df_balanced_indexed.select("ALT_Blood","GGT_Blood","Glucose_Blood",
                                    "Creatine_Blood","THC_Positive_Urine",
                                    "Cocaine_Positive_Urine",
                                    "Opioid_Positive_Urine","Urine_pH",
                                    "Attendance","Marks_drop",
                                    "Assignment_delays","Frequent_leave",
                                    "Mood_Swings","Peer_Group_Issues",
                                    "Disciplinary_Actions","Counseling_Referral",
                                    "Social_Media_Overuse","Gaming_Addiction",
                                    "Addiction_Label")

## Convert cat_df pyspark dataframe to pandas

  cat_df_pandas = cat_df.toPandas()
# Step 1: Detect string (categorical) columns in cat_df_pandas

  cat_features = [
      col for col in cat_df_pandas.select_dtypes(include=['object', 'category']) \
      .columns if col != "Addiction_Label"
  ]

  print("Categorical features in cat_df_pandas detected:")
  print(cat_features)

# Categorical features in cat_df_pandas detected:
# ['THC_Positive_Urine', 'Cocaine_Positive_Urine',
# 'Opioid_Positive_Urine', 'Addiction_Label']

# Prepare dataset (X = features, y = labels)

  X_features = cat_df_pandas.drop("Addiction_Label", axis=1)
  y_label = cat_df_pandas["Addiction_Label"]

  print("Columns in X_features:", X_features.columns.tolist())

# You split all features (including categorical ones)
# test_size=0.3, means 30 % for testing and 70 % for training

  X_train, X_test, y_train, y_test = train_test_split(X_features, y_label, \
                                                      test_size=0.3)

# Tell CatBoost which features (by name) in X_train are categorical

# Initialize model with accuracy as metric

  cat_model = CatBoostClassifier(
      cat_features=cat_features,
      eval_metric='Accuracy',
      iterations=100,           # lower than 500
      # iterations=10,           # lower than 500
      depth=5,                  # shallow trees
      learning_rate=0.05,       # slower learning
      random_state=42,
      verbose=100
      )

# Train with eval_set to get accuracy

  cat_model.fit(
      X_train, y_train,
      eval_set=(X_test, y_test)
      )

# Save the model, Overwrites if already present

  # cat_model.save_model('/content/Student-Drug-Addiction-Prediction/drive/MyDrive/model_drug_addiction.cbm')

  cat_model.save_model('/content/Student-Drug-Addiction-Prediction/model_drug_addiction_tmp.cbm')

# Step 1: Make predictions

  predictions = cat_model.predict(X_test)

# Accuracy / Precision / Recall / F1

  print()
  print()

  print("Over all Accuracy:", accuracy_score(y_test, predictions))

  print("Precision (macro):",
        precision_score(y_test, predictions, average='macro'))

  print("Recall (macro):",
        recall_score(y_test, predictions, average='macro'))

  print("F1 Score:",
        f1_score(y_test, predictions, average='macro'))

  from sklearn.metrics import confusion_matrix

  cm = confusion_matrix(y_test, predictions)

  print("Confusion Matrix:")
  print(cm)

# 6️⃣ Classification Report (Recommended)

  from sklearn.metrics import classification_report
  print()
  print("Classification Report ....")
  print()
  print(classification_report(y_test, predictions))


# Step 2: Combine predictions with actual values and inputs
# Convert to Pandas Dataframe

  X_test = pd.DataFrame(X_test)
  y_test = pd.Series(y_test)

  results_df = X_test.reset_index(drop=True).copy()

  results_df["Actual"] = y_test.reset_index(drop=True)
  results_df["Predicted"] = predictions.flatten()

  print("Final output with Prediction Column....")
  print(results_df.head(10).to_string())


## Accuracy Evaluation starts here, metrices are, accuracy, precision, f1-score,
## AUC, Confusion matrix

## OUTPUT - Accuracy of CATBOOST

# bestTest = 0.9895418327
# bestIteration = 99

# Cross Validator Accuracy Mean: 0.9897658484438772

# AUC (OvR) :- Belongs to Computational Mathematics, that is,
# Numerical integration (e.g., trapezoidal rule, Newton-Raphson methods)

## Evaluation Summary of all the above Metrices used :-

# macro - Average taken from all classes, here 'Addicted' /
# 'Not Addicted' are the classes, multiply by 100 will get correct percent

# Cross Validator Accuracy Mean: 0.9897658484438772 --> 98.97 %
# Cross-Validation Precision (macro) Mean: 0.9911029449923652 --> 99.11 %
# Cross-Validation Recall (macro) Mean: 0.9880983378221118 --> 98.80 %
# F1 Score (macro): 0.9895029583018505 --> 98.95 %
# AUC (Area Under Curve): 0.9999459799851371 --> 99.99 %
#############

  display(HTML("<h3 style='color:magenta; text-align:left;'> \
    Model Building Completed..</h3>"))

  print()
  go_back_function(event)

def single_input_and_predict(event):
  print()

  clear_result_grid()

  filename = '/content/Student-Drug-Addiction-Prediction/Single_Input_Drug.csv'

  headers = ["Name", "ALT_Blood", "GGT_Blood",
              "Glucose_Blood", "Creatine_Blood",
              "THC_Positive_Urine", "Cocaine_Positive_Urine",
              "Opioid_Positive_Urine", "Urine_pH",
              "Attendance", "Marks_drop",
              "Assignment_delays", "Frequent_leave",
              "Mood_Swings", "Peer_Group_Issues",
              "Disciplinary_Actions", "Counseling_Referral",
              "Social_Media_Overuse", "Gaming_Addiction"
            ]

  css = """
      <style>
          .green-text .widget-label {
              color: green !important;
              font-size: 18px;
          }
      </style>
      """

# Define the input box or frame style

####### background-colors are

# "lightgray', "whitesmoke", "lavender", "mintcream", "aliceblue", "linen"
# "ivory", "honeydew", "floralwhite", "ghostwhite"


########### Define the CSS class with the desired style for frame

# In the code given below, the f-string is used to define the custom CSS (Cascading Style Sheets)
# for an input box. The variable input_box_style is expected to contain the CSS styling properties
# for the input box. The f-string allows the variable's value to be inserted directly into the string,
# resulting in a dynamic CSS style applied to the input box.

# The following display(HTML(css)), activate all css declarations

  display(HTML(css))

  display(HTML("<h2 style='color:magenta; text-align:center;'>Input your Details</h2>"))
  print()
  print()

  Name     		 		= widgets.Text(description	=
                          'Patient Name>>>>>>>>>>>>>>>',
                          style={'description_width': 'initial'},
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  ALT_Blood		 		= widgets.IntText(description	=
                          'ALT in Blood[20 to 150]>>>>>>>>',
                          style={'description_width': 'initial'},
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  GGT_Blood		 		= widgets.IntText(description	=
                          'GGT in Blood[20 to 200]>>>>>>>>',
                          style={'description_width': 'initial'},
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Glucose_Blood	 		= widgets.IntText(description	=
                          'Glucode Level in Blood[80 to 300]>',
                          style={'description_width': 'initial'},
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Creatine_Blood 		= widgets.IntText(description	=
                          'Creatine Level in Blood[0.5 to 3.0]>',
                          style={'description_width': 'initial'},
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Yes_No_options = ["No","Yes"]

  THC_Positive_Urine	 = widgets.Dropdown(description	=
                          'THC Positive in Urine>>>>>>',
                          style={'description_width': 'initial'},
                          options=Yes_No_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Cocaine_Positive_Urine = widgets.Dropdown(description	=
                          'Cocaine Positive Urine>>>>>',
                          style={'description_width': 'initial'},
                          options=Yes_No_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Opioid_Positive_Urine = widgets.Dropdown(description	=
                          'Opiod Positive   Urine>>>>>',
                          style={'description_width': 'initial'},
                          options=Yes_No_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Urine_pH		 		= widgets.FloatText(description	=
                          'Urine PH (Min=3.0,Max=8.0)>',
                          style={'description_width': 'initial'},
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  score_options = [0,1]

  Attendance   			 = widgets.Dropdown(description	=
                          'Attendance Drop(0-No,1-Yes>',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Marks_drop   			 = widgets.Dropdown(description	=
                          'Marks Drop(0-No, 1-Yes)>>>>',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Assignment_delays		 = widgets.Dropdown(description	=
                          'Assignment Delays(0-No,1-Ye',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Frequent_leave		 = widgets.Dropdown(description	=
                          'Frequent Leave(0-No,1-Yes)>',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Mood_Swings			 = widgets.Dropdown(description	=
                          'Mood Swings(0-No, 1-Yes)>>>',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Peer_Group_Issues		 = widgets.Dropdown(description	=
                          'Peer Group Issues(0-No,1-Ye',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Disciplinary_Actions	 = widgets.Dropdown(description	=
                          'Disciplinary Actions(0-No,>>)',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Counseling_Referral	 = widgets.Dropdown(description	=
                          'Counseling Regerral(0-No>>)',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Social_Media_Overuse	 = widgets.Dropdown(description	=
                          'Social Media Overuse(0-No>>)',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))

  Gaming_Addiction		 = widgets.Dropdown(description	=
                          'Gaming Addiction(0-No,1-Yes)',
                          style={'description_width': 'initial'},
                          options=score_options,
                          layout=widgets.Layout(
                          width='400px',
                          border='2px solid blue',
                          border_radius='5px',
                          padding='1px'))


# Add CSS classes declared above to the input widgets

  Name.add_class('green-text')
  ALT_Blood.add_class('green-text')
  GGT_Blood.add_class('green-text')
  Glucose_Blood.add_class('green-text')
  Creatine_Blood.add_class('green-text')
  THC_Positive_Urine.add_class('green-text')
  Cocaine_Positive_Urine.add_class('green-text')
  Opioid_Positive_Urine.add_class('green-text')
  Urine_pH.add_class('green-text')
  Attendance.add_class('green-text')
  Marks_drop.add_class('green-text')
  Assignment_delays.add_class('green-text')
  Frequent_leave.add_class('green-text')
  Mood_Swings.add_class('green-text')
  Peer_Group_Issues.add_class('green-text')
  Disciplinary_Actions.add_class('green-text')
  Counseling_Referral.add_class('green-text')
  Social_Media_Overuse.add_class('green-text')
  Gaming_Addiction.add_class('green-text')

# Display Data Entry headings and comments
# The following comment statements guide the customer to enter values correctly

  Name_container = widgets.HBox(children=[Name])
  ALT_container = widgets.HBox(children=[ALT_Blood])
  GGT_container = widgets.HBox(children=[GGT_Blood])
  Glucose_container = widgets.HBox(children=[Glucose_Blood])
  Creatine_container = widgets.HBox(children=[Creatine_Blood])
  THC_Positive_container = widgets.HBox(children=[THC_Positive_Urine])
  Cocaine_container = widgets.HBox(children=[Cocaine_Positive_Urine])
  Opioid_container = widgets.HBox(children=[Opioid_Positive_Urine])
  Urine_pH_container = widgets.HBox(children=[Urine_pH])
  Attendance_container = widgets.HBox(children=[Attendance])
  Marks_container = widgets.HBox(children=[Marks_drop])
  Assignment_container = widgets.HBox(children=[Assignment_delays])
  Frequent_leave_container = widgets.HBox(children=[Frequent_leave])
  Mood_container = widgets.HBox(children=[Mood_Swings])
  Peer_Group_container = widgets.HBox(children=[Peer_Group_Issues])
  Disciplinary_container = widgets.HBox(children=[Disciplinary_Actions])
  Counseling_container = widgets.HBox(children=[Counseling_Referral])
  Social_Media_container = widgets.HBox(children=[Social_Media_Overuse])
  Gaming_container = widgets.HBox(children=[Gaming_Addiction])

  display(Name_container)
  display(ALT_container)
  display(GGT_container)
  display(Glucose_container)
  display(Creatine_container)
  display(THC_Positive_container)
  display(Cocaine_container)
  display(Opioid_container)
  display(Urine_pH_container)
  display(Attendance_container)
  display(Marks_container)
  display(Assignment_container)
  display(Frequent_leave_container)
  display(Mood_container)
  display(Peer_Group_container)
  display(Disciplinary_container)
  display(Counseling_container)
  display(Social_Media_container)
  display(Gaming_container)

  print()
  print()

# Apply HTML/CSS for the above buttons

  custom_css_single = """
    <style>
      .custom-button-single {
          width: 250px;
          height: 50px;
          color: blue; /* Text color */
          background: linear-gradient(to right, pink, magenta); /* Gradient from yellow to orange */
          border: 2px solid black;
          transition: background 0.3s ease; /* Smooth transition for hover effect */
      }

/* Add a mouse hover effect */
      .custom-button-single:hover {
          background: linear-gradient(to right, white, lightgray); /* Gradient changes on hover */
      }
      </style>

    """

# Display Main Heading

  display(HTML("<h2 style='color:green; text-align:center;'>Prediction using Single Input</h2>"))
  print()

# Declare your buttons

  save_button_single = widgets.Button(description="Save Data")
  predict_button_single = widgets.Button(description="Predict Result")
  exit_button_single = widgets.Button(description="Go Back to Main Menu")

# # Attach the above style sheet to each buttons

  save_button_single.add_class("custom-button-single")
  predict_button_single.add_class("custom-button-single")
  exit_button_single.add_class("custom-button-single")

# Display the custom button

  display(HTML(custom_css_single))

  buttons_horizontal_layout_single = widgets.HBox([save_button_single,predict_button_single,
                                              exit_button_single])

# # Align the buttons using the following

  align_layout_single = widgets.Layout(justify_content='center')

# Arrange buttons horizontally, in center of the output area

  left_container_single = widgets.Box([buttons_horizontal_layout_single],
                                      layout=align_layout_single)

  display(left_container_single)
  print()

# Declare the following 'on_button_clicked' function inside
# the outer function 'single_input_and_predict'

  def on_button_clicked(save_button_single):
    print()

# Assign values from text boxes to variables
    entered_Name = Name.value
    entered_ALT = ALT_Blood.value
    entered_GGT = GGT_Blood.value
    entered_Glucose = Glucose_Blood.value
    entered_Creatine = Creatine_Blood.value
    entered_THC_Positive = THC_Positive_Urine.value
    entered_Cocaine = Cocaine_Positive_Urine.value
    entered_Opioid = Opioid_Positive_Urine.value
    entered_Urine_pH = Urine_pH.value
    entered_Attendance = Attendance.value
    entered_Marks_drop = Marks_drop.value
    entered_Assignment = Assignment_delays.value
    entered_Frequent_leave = Frequent_leave.value
    entered_Mood_Swings = Mood_Swings.value
    entered_Peer_Group = Peer_Group_Issues.value
    entered_Disciplinary = Disciplinary_Actions.value
    entered_Counseling = Counseling_Referral.value
    entered_Social_Media = Social_Media_Overuse.value
    entered_Gaming = Gaming_Addiction.value

# Save your data as csv file 'Single_Input_Autism.csv'
    import csv
    with open(filename, 'w', newline='') as f:
          writer = csv.writer(f)
          writer.writerow(headers)
          writer.writerow([entered_Name,
                            entered_ALT,
                            entered_GGT,
                            entered_Glucose,
                            entered_Creatine,
                            entered_THC_Positive,
                            entered_Cocaine,
                            entered_Opioid,
                            entered_Urine_pH,
                            entered_Attendance,
                            entered_Marks_drop,
                            entered_Assignment,
                            entered_Frequent_leave,
                            entered_Mood_Swings,
                            entered_Peer_Group,
                            entered_Disciplinary,
                            entered_Counseling,
                            entered_Social_Media,
                            entered_Gaming
                           ])

# modify print statement
          print('Data has been written to', filename)

# # Save your data as csv file 'Single_Air_Quality.csv'

  save_button_single.on_click(on_button_clicked)
  predict_button_single.on_click(predict_single)
  exit_button_single.on_click(go_back_function_1)

def predict_single(event):
  clear_result_grid()
  print()
  display(HTML("<h3 style='color:magenta; text-align:left;'> \
    Single Input Prediction Starting Please...wait few Seconds !!!</h3>"))

# # Read Single Input Data for prediction

## Convert BULK CSV FILE to pandas Dataframe

  single_df_pandas = pd.read_csv("/content/Student-Drug-Addiction-Prediction/Single_Input_Drug.csv")

# # ## CROSS VERIFICATION
#   print("Display few Rows in single_df_pandas")
#   print(single_df_pandas.head(5).to_string(index=False))

# Drop the Name col
  single_cat_features = single_df_pandas.drop(columns=['Name'])

## LOAD the saved CAT BOOST model

# Initialize the model class
  loaded_model = CatBoostClassifier()

# Load the saved model
  loaded_model.load_model('/content/Student-Drug-Addiction-Prediction/model_drug_addiction.cbm')

# Step 1: Make predictions
  single_predictions = loaded_model.predict(single_cat_features)

# Step 2: Combine predictions with actual values and inputs
  results_df = single_df_pandas.copy()
  results_df["Prediction"] = single_predictions

# # Step 3: Print or view the result
#  print(results_df.head(5).to_string(index=False))

# ## Extract only selected cols for final display

  single_selected = results_df[["Name", "Prediction", "ALT_Blood", "GGT_Blood",
                                    "Glucose_Blood", "Creatine_Blood",
                                    "THC_Positive_Urine", "Cocaine_Positive_Urine",
                                    "Opioid_Positive_Urine", "Urine_pH",
                                    "Attendance", "Marks_drop",
                                    "Assignment_delays", "Frequent_leave",
                                    "Mood_Swings", "Peer_Group_Issues",
                                    "Disciplinary_Actions", "Counseling_Referral",
                                    "Social_Media_Overuse", "Gaming_Addiction"
                              ]]

# # background colors, 'red','green','blue','yellow','orange','purple','pink'

  styled_df = (
      single_selected.style
      .format({col: "{:.2f}" for col in single_selected.select_dtypes(include="float").columns})
      .map(lambda x: 'background-color: lightcyan',
          subset=["Name", "Prediction", "ALT_Blood", "GGT_Blood",
                  "Glucose_Blood", "Creatine_Blood",
                  "THC_Positive_Urine", "Cocaine_Positive_Urine",
                  "Opioid_Positive_Urine", "Urine_pH",
                  "Attendance", "Marks_drop",
                  "Assignment_delays", "Frequent_leave",
                  "Mood_Swings", "Peer_Group_Issues",
                  "Disciplinary_Actions", "Counseling_Referral",
                  "Social_Media_Overuse", "Gaming_Addiction"])
              )

# # Generate an HTML table(html_table) with customized CSS
# # 'th' = Table Heading, 'td' = Table Data

  html_table = styled_df.hide(axis="index").set_table_styles(
        [{'selector': 'th', 'props':[('color', 'red'), ('padding', '5px')]},
         {'selector': 'td', 'props': [('vertical-align', 'middle'),('padding', '5px')]}
        ]).to_html()

# # Draw a frame around your output and Display the contents

# # the f prefix enables you to embed expressions within a string by enclosing them in curly braces {}
# # Within the string, you have embedded an expression {html_table}.
# # The value of the html_table variable is dynamically inserted into the string at that location, that
# # values are nothing but your prediction results with headings.

  output_frame = f"""
      <div style="border: 2px solid black; padding: 10px;width: 270%; background-color: lightgrey;">
          <h3 style='color:blue; text-align:center;'>Final Prediction as per your Input</h3>
          {html_table}
      </div>
      """
  print()
  print()
  print()
  display(HTML(output_frame))

# 'Click Here to go back' button style, after prediction

  custom_css_1 = """
    <style>
    .custom-button-1 {
        width: 200px;  /* Change this value for width */
        height: 50px;  /* Change this value for height */
        color: blue; /* Text color */

        border: 2px solid black;
        border-radius: 10px; /* Rounded corners */
        background: linear-gradient(to right, lightpink, deeppink); /* Gradient background */
        transition: width 1s; /* Delay for Button Stretch */
    }

  /* Add a mouse hover effect */

    .custom-button-1:hover {
        background: linear-gradient(to right, white, white, white);
        width: 250px; /* Button slowly Stretch horizontally */

        }
    .container {
      overflow: hidden; /* Hide content that overflows the container */
    }
    </style>

    """
  button = widgets.Button(description="Click Here to Go Back")
  button.add_class("custom-button-1")
  display(HTML(custom_css_1))
  button.on_click(on_button_click_goback)

# Display the button
  display(widgets.HBox([button]))
  print()
  print()
  print()

def on_button_click_goback(event):
  single_input_and_predict(event)

def go_back_function(event):
#  clear_result_grid()
  main_function()
  print()

def go_back_function_1(event):
  clear_result_grid()
  main_function()

  print()

def bulk_on_button_click_goback(event):
  bulk_data_load(event)

def bulk_data_load(event):
  print()
  clear_result_grid()
  set_focus_cell2()
# Declare your upload buttons

  browse_button = widgets.Button(description="Upload Bulk Data")
  predict_button = widgets.Button(description="Predict Result")
  exit_button = widgets.Button(description="Go Back to Main Menu")

# Apply HTML/CSS for the above buttons

  custom_css_bulk = """
      <style>
        .custom-button-bulk {
            width: 250px;
            height: 50px;
            color: blue; /* Text color */
            background: linear-gradient(to right, lightgreen, pink); /* Gradient from yellow to orange */
            border: 2px solid black;
            transition: background 0.3s ease; /* Smooth transition for hover effect */
        }

  /* Add a mouse hover effect */
        .custom-button-bulk:hover {
            background: linear-gradient(to right, white, lightgray); /* Gradient changes on hover */
        }
        </style>

    """
# Attach the above style sheet to each buttons

  browse_button.add_class("custom-button-bulk")
  predict_button.add_class("custom-button-bulk")
  exit_button.add_class("custom-button-bulk")

# Display the custom button

  display(HTML(custom_css_bulk))

  buttons_horizontal_layout_1 = widgets.HBox([browse_button,predict_button,exit_button])

# # Align the buttons using the following

  align_layout = widgets.Layout(justify_content='center')

# Arrange buttons horizontally, in center of the output area

  left_container_1 = widgets.Box([buttons_horizontal_layout_1], layout=align_layout)

  print()
  print()
  print()
  display(HTML("<h3 style='color:magenta; text-align:center;'>Bulk Input Data Prediction Sub Menu</h3>"))
  print()
  display(left_container_1)
  print()
  print()

  def upload_file_fn(event):
      selected_file = files.upload()
      for dataset_name in selected_file.keys():
            source_path = f"/content/Student-Drug-Addiction-Prediction/{dataset_name}"
#            destination_path = f"/content/drive/MyDrive/{dataset_name}"
            # !mv "$source_path" "$destination_path"

  browse_button.on_click(upload_file_fn)
  predict_button.on_click(bulk_predict_fn)
  exit_button.on_click(go_back_function_1)

# 'exit_function' is used separately because you cannot call 'main_function()' directly
# in on_click event. '0 argument expected but 1 is given' error will be flagged

def bulk_predict_fn(event):

# Read Bulk Input Data for prediction

  clear_result_grid()
  print()
  display(HTML("<h3 style='color:red; text-align:left;'> \
    Bulk Input Data Prediction starting this may take 1 to 2 minutes... Please...wait !!!</h3>"))

## Since Bulk_Drug_Data.csv will only contains data relatively small in size
## there is no need to convert it using PYSPARK. Instead you can directly use
## PANDAS

## Convert BULK CSV FILE to pandas Dataframe

  bulk_df_pandas = pd.read_csv("/content/Student-Drug-Addiction-Prediction/Bulk_Drug_Data.csv")

# ## CROSS VERIFICATION
#   print("Display few Rows in bulk_df_pandas")
#   print(bulk_df_pandas.head(5).to_string(index=False))

# Display few Rows in bulk_df_pandas
#       Name  ALT_Blood  GGT_Blood  Glucose_Blood  Creatine_Blood THC_Positive_Urine Cocaine_Positive_Urine Opioid_Positive_Urine  Urine_pH  Attendance  Marks_drop  Assignment_delays  Frequent_leave  Mood_Swings  Peer_Group_Issues  Disciplinary_Actions  Counseling_Referral  Social_Media_Overuse  Gaming_Addiction
#     JOSE-A         46         82            121            1.60                 No                     No                    No       5.4           0           0                  0               0            1                  1                     1                    1                     1                 1
#   DELNA-NA         57         27             93            1.09                 No                     No                    No       5.9           0           0                  0               1            0                  0                     0                    0                     1                 0
#  THOMAS-NA         92         57             70            1.07                 No                     No                    No       6.0           0           0                  1               0            0                  1                     1                    1                     1                 0
#  RASHEED-A         43        118             70            0.89                Yes                     No                    No       6.0           0           0                  1               0            0                  1                     0                    1                     0                 1
# AVINASH-NA         49        137            112            1.09                Yes                    Yes                   Yes       6.7           0           0                  1               0            0                  1                     0                    0                     0                 0

# Drop the Name col
  bulk_cat_features = bulk_df_pandas.drop(columns=['Name'])

## LOAD the saved CAT BOOST model

# Initialize the model class
  loaded_model = CatBoostClassifier()

# Load the saved model
  loaded_model.load_model('/content/Student-Drug-Addiction-Prediction/model_drug_addiction.cbm')

# Step 1: Make predictions
  bulk_predictions = loaded_model.predict(bulk_cat_features)

# Step 2: Combine predictions with actual values and inputs
  results_df = bulk_df_pandas.copy()

  results_df["Prediction"] = bulk_predictions

# # Step 3: Print or view the result
#   print(results_df.head(25).to_string(index=False))

# ## Extract only selected cols for final display

  bulk_selected = results_df[["Name", "Prediction", "ALT_Blood", "GGT_Blood",
                                    "Glucose_Blood", "Creatine_Blood",
                                    "THC_Positive_Urine", "Cocaine_Positive_Urine",
                                    "Opioid_Positive_Urine", "Urine_pH",
                                    "Attendance", "Marks_drop",
                                    "Assignment_delays", "Frequent_leave",
                                    "Mood_Swings", "Peer_Group_Issues",
                                    "Disciplinary_Actions", "Counseling_Referral",
                                    "Social_Media_Overuse", "Gaming_Addiction"
                              ]]
## CROSS VERIFICATION

  # print("Final Result with Selected Columns .....")
  # print(bulk_selected.head(25).to_string(index=False))

# # ########## Display prediction results in color frame

# # Apply color to the DataFrame output

# # The applymap method is used to apply a function to each element of the DataFrame
# # the lambda function in this code snippet is used to assign the CSS style 'color: red'
# # to the cells in the 'Price' column of the DataFrame when generating its styled representation.

# # background colors, 'red','green','blue','yellow','orange','purple','pink'

  styled_df = (
      bulk_selected.style
      .format({col: "{:.2f}" for col in bulk_selected.select_dtypes(include="float").columns})
      .map(lambda x: 'background-color: white',
          subset=["Name", "Prediction", "ALT_Blood", "GGT_Blood",
                  "Glucose_Blood", "Creatine_Blood",
                  "THC_Positive_Urine", "Cocaine_Positive_Urine",
                  "Opioid_Positive_Urine", "Urine_pH",
                  "Attendance", "Marks_drop",
                  "Assignment_delays", "Frequent_leave",
                  "Mood_Swings", "Peer_Group_Issues",
                  "Disciplinary_Actions", "Counseling_Referral",
                  "Social_Media_Overuse", "Gaming_Addiction"])
              )

# # Generate an HTML table(html_table) with customized CSS
# # 'th' = Table Heading, 'td' = Table Data

  html_table = styled_df.hide(axis="index").set_table_styles(
        [{'selector': 'th', 'props':[('color', 'red'), ('padding', '5px')]},
         {'selector': 'td', 'props': [('vertical-align', 'middle'),('padding', '5px')]}
        ]).to_html()

# # Draw a frame around your output and Display the contents

# # the f prefix enables you to embed expressions within a string by enclosing them in curly braces {}
# # Within the string, you have embedded an expression {html_table}.
# # The value of the html_table variable is dynamically inserted into the string at that location, that
# # values are nothing but your prediction results with headings.

  output_frame = f"""
      <div style="border: 2px solid black; padding: 10px;width: 270%; background-color: lightgrey;">
          <h3 style='color:blue; text-align:center;'>Final Prediction as per your Input</h3>
          {html_table}
      </div>
      """
  print()
  print()
  print()
  display(HTML(output_frame))

# # ####### Show Predictions ENDS HERE

#   print()
#   print()

# # Add a Button 'Click Here to go back' below the output after prediction

  custom_css_bulk = """
    <style>
    .custom-button-bulk {
        width: 200px;  /* Change this value for width */
        height: 50px;  /* Change this value for height */
        color: blue; /* Text color */

        border: 2px solid black;
        border-radius: 10px; /* Rounded corners */
        background: linear-gradient(to right, lightpink, deeppink); /* Gradient background */
        transition: width 1s; /* Delay for Button Stretch */
    }

  /* Add a mouse hover effect */

    .custom-button-bulk:hover {
        background: linear-gradient(to right, white, white, white);
        width: 250px; /* Button slowly Stretch horizontally */

        }
    .container {
      overflow: hidden; /* Hide content that overflows the container */
    }
    </style>

    """

  button_bulk = widgets.Button(description="Click Here to Go Back")
  button_bulk.add_class("custom-button-bulk")

# Apply CSS styles
  display(HTML(custom_css_bulk))
# Display the button
  display(button_bulk)
  button_bulk.on_click(bulk_data_load)

############## Bulk user Input Prediction ended here

def main_function():

# clear_result_grid()
# Set focus to result grid

  set_focus_cell2()
  print()

  display(HTML("<h3 style='color:black;  ; text-align:center;'>AddictAid - Control Console</h3>"))
  print()

#### Create your button names for each events

# First Row button names

  load_data_button = widgets.Button(description="1. Load Data")
  eda_analysis_button = widgets.Button(description="2. EDA Analysis")
  pre_process_button = widgets.Button(description="3. Pre Process Data")
# Second Row button names

  transform_data_button = widgets.Button(description="4. Transform Data")
  build_model_and_save_button = widgets.Button(description="5. Build Model & Save")
  single_mail_predict_button = widgets.Button(description="6. Prediction(Single Input)")
# Third Row button names

  bulk_mail_predict_button = widgets.Button(description="7. Prediction(Bulk Input)")

# Assign Click Event for each python Functions declared above

  load_data_button.on_click(load_data)
  eda_analysis_button.on_click(EDA_start)
  pre_process_button.on_click(pre_process_data)

  transform_data_button.on_click(transformation)
  build_model_and_save_button.on_click(build_model_save)
  single_mail_predict_button.on_click(single_input_and_predict)

  bulk_mail_predict_button.on_click(bulk_data_load)

  menu_button_css = """
  <style>
  body {
    background-image: url('');
    background-size: cover;
    background-repeat: no-repeat;
    background-position: center;
    margin: 0;
    padding: 0;
    }
  .custom-button {
      width: 250px;
      height: 100px;
      border-radius: 50%;
      background: linear-gradient(135deg, #667eea, #764ba2);
      color: white;
      border: none;
      font-size: 16px;
      font-weight: bold;
      cursor: pointer;
      transition: all 0.3s ease;
      box-shadow: 0 5px 15px rgba(0,0,0,0.3);
  }

  .custom-button:hover {
      background: linear-gradient(135deg, #ff6a00, #ee0979);
      transform: scale(1.1) rotate(5deg);
  }

  </style>

  """

#### Button special effects ends here

### Apply the above CSS class properties on top of button names

  load_data_button.add_class("custom-button")
  eda_analysis_button.add_class("custom-button")
  pre_process_button.add_class("custom-button")
  transform_data_button.add_class("custom-button")
  build_model_and_save_button.add_class("custom-button")
  single_mail_predict_button.add_class("custom-button")
  bulk_mail_predict_button.add_class("custom-button")

#### Important : Activate the custom CSS on buttons

  display(HTML(menu_button_css))

# Arrange buttons horizontally, HBox is used to display buttons horizontally
#                               VBox is used to display buttons vertically

  buttons_horizontal_layout_1 = widgets.HBox([load_data_button,
                                              eda_analysis_button, pre_process_button])
  buttons_horizontal_layout_2 = widgets.HBox([transform_data_button,
                                              build_model_and_save_button,
                                              single_mail_predict_button])
  buttons_horizontal_layout_3 = widgets.HBox([bulk_mail_predict_button])

# Center the buttons using a centered container in result grid

  centered_layout = widgets.Layout(justify_content='center')

# Arrange buttons horizontally, in center of the output area

  centered_container_1 = widgets.Box([buttons_horizontal_layout_1], layout=centered_layout)
  centered_container_2 = widgets.Box([buttons_horizontal_layout_2], layout=centered_layout)
  centered_container_3 = widgets.Box([buttons_horizontal_layout_3], layout=centered_layout)

  # Display the centered buttons with one line gap between each row

  display(centered_container_1)
  print()
  display(centered_container_2)
  print()
  display(centered_container_3)
  print()

### Display your Marquee

  html_content = """
  <div class="scrolling-message-container">
    <marquee behavior="scroll" direction="left" scrollamount="10" style="color: red;">
    Designed and Developed by: XXXXXXX</marquee>
  </div>
  """
  scroll_msg = HTML(html_content)
  display(scroll_msg)

######## End of Main Menu Design

# Declare a main function as follows to invoke all other function

if __name__ == "__main__":

# Call main function

  main_function()
